# # Convertir el diccionario en una especificacion tecnica de la extension

La pregunta ya no será “¿existe la variable?”, sino:

“¿Qué dato crudo debe capturar mi extensión para poder calcular esta variable exactamente?”

In [19]:
from pathlib import Path

import pandas as pd
import numpy as np

In [20]:
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 250)

REPORTS_DIR = Path("../reports/tables").resolve()
CONFIGS_DIR = Path("../configs").resolve()

CONFIGS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

dictionary = pd.read_csv(
    REPORTS_DIR
    / "feature_dictionary_validation.csv"
)

print(dictionary.shape)

display(
    dictionary.head()
)

(48, 10)


,variable,categoria,rol_ml,a11y_estado,repo01_estado,delfi_estado,irr_aporte,requiere_experto,captura_extension,estado_final
0,instance_id,CONTEXTO,IDENTIFICADOR,DIRECTA_O_DERIVADA,NO,NO,NO,False,DIRECTA,RESPALDADA_POR_DATOS_PUBLICOS
1,platform,CONTEXTO,CONTEXTO,NO,NO,NO,NO,False,DIRECTA,RECOLECCION_PROPIA
2,task_type,CONTEXTO,CONTEXTO,DIRECTA_O_DERIVADA,NO,NO,NO,False,DIRECTA,RESPALDADA_POR_DATOS_PUBLICOS
3,page_type,CONTEXTO,CONTEXTO,NO,NO,NO,NO,False,DIRECTA,RECOLECCION_PROPIA
4,task_time_s,TEMPORAL,X,DIRECTA_O_DERIVADA,NO,NO,NO,False,DIRECTA,RESPALDADA_POR_DATOS_PUBLICOS


In [21]:
identificadores = dictionary[
    dictionary["rol_ml"] == "IDENTIFICADOR"
].copy()

contexto = dictionary[
    dictionary["rol_ml"] == "CONTEXTO"
].copy()

features_x = dictionary[
    dictionary["rol_ml"] == "X"
].copy()

targets_y = dictionary[
    dictionary["rol_ml"] == "Y"
].copy()

print("Identificadores:", len(identificadores))
print("Contexto:", len(contexto))
print("X:", len(features_x))
print("Y:", len(targets_y))

Identificadores: 1
Contexto: 3
X: 40
Y: 4


In [22]:
def spec(
    variable,
    raw_signal,
    mechanism,
    calculation,
    unit,
    granularity,
    storage,
    notes=""
):
    return {
        "variable": variable,
        "raw_signal": raw_signal,
        "capture_mechanism": mechanism,
        "calculation": calculation,
        "unit": unit,
        "granularity": granularity,
        "raw_storage": storage,
        "notes": notes
    }

In [23]:
specifications = [
    
    spec(
        "instance_id",
        "Identificador generado al iniciar una ejecución",
        "crypto.randomUUID()",
        "Se genera una vez por ejecución usuario-tarea",
        "UUID",
        "EJECUCION",
        "sessions",
        "No se usa como predictor."
    ),

    spec(
        "platform",
        "hostname / origen de la página",
        "window.location.hostname",
        "Normalización del dominio",
        "categorica",
        "EJECUCION",
        "sessions"
    ),

    spec(
        "task_type",
        "Identificador/tipo de tarea experimental",
        "Configuración de la sesión de evaluación",
        "Asignado al iniciar la tarea",
        "categorica",
        "EJECUCION",
        "sessions"
    ),

    spec(
        "page_type",
        "URL + estructura de página",
        "window.location + clasificación",
        "Clasificación de página según reglas de la plataforma",
        "categorica",
        "PAGINA",
        "page_states",
        "No debe inferirse únicamente del pathname sin validación."
    )
]

In [24]:
specifications += [

    spec(
        "task_time_s",
        "task_start_timestamp, task_end_timestamp",
        "performance.now() / Date.now()",
        "(task_end - task_start) / 1000",
        "s",
        "EJECUCION",
        "sessions"
    ),

    spec(
        "mean_response_time_ms",
        "requestStart, responseEnd",
        "PerformanceObserver / Resource Timing API",
        "media(responseEnd - requestStart)",
        "ms",
        "EJECUCION",
        "performance_events",
        "Tiempo de respuesta técnico de recursos/red; no confundir con feedback de interfaz."
    ),

    spec(
        "max_response_time_ms",
        "requestStart, responseEnd",
        "PerformanceObserver / Resource Timing API",
        "max(responseEnd - requestStart)",
        "ms",
        "EJECUCION",
        "performance_events"
    ),

    spec(
        "mean_pre_action_pause_s",
        "timestamps consecutivos de acciones",
        "Listeners de interacción",
        "media(timestamp_accion - timestamp_evento_previo) / 1000",
        "s",
        "EJECUCION",
        "interaction_events",
        "Es una pausa conductual; no debe interpretarse directamente como confusión."
    )
]

In [25]:
specifications += [

    spec(
        "click_count",
        "eventos click",
        "document.addEventListener('click', ...)",
        "conteo de eventos click válidos",
        "eventos",
        "EJECUCION",
        "interaction_events"
    ),

    spec(
        "keypress_count",
        "eventos keydown",
        "document.addEventListener('keydown', ...)",
        "conteo de pulsaciones",
        "eventos",
        "EJECUCION",
        "interaction_events",
        "No almacenar el contenido escrito por razones de privacidad; solo metadatos necesarios."
    ),

    spec(
        "scroll_count",
        "eventos/episodios de scroll",
        "document.addEventListener('scroll', ...)",
        "agrupar eventos consecutivos de scroll en episodios",
        "episodios",
        "EJECUCION",
        "interaction_events"
    ),

    spec(
        "backtracking_count",
        "secuencia de estados/URLs visitados",
        "history/navigation + page state logger",
        "conteo de regreso a estados previamente visitados",
        "eventos",
        "EJECUCION",
        "navigation_events"
    ),

    spec(
        "repeated_action_count",
        "tipo, selector, timestamp y posición de acciones",
        "listeners de interacción",
        "acciones equivalentes repetidas dentro de una ventana temporal definida",
        "eventos",
        "EJECUCION",
        "interaction_events",
        "La ventana temporal deberá fijarse mediante piloto; no equivale automáticamente a error."
    ),

    spec(
        "navigation_steps",
        "cambios de estado/página",
        "History API + URL + observación DOM",
        "conteo de transiciones de navegación",
        "pasos",
        "EJECUCION",
        "navigation_events"
    )
]

In [26]:
specifications += [

    spec(
        "feedback_present",
        "cambios DOM posteriores a una acción",
        "MutationObserver",
        "1 si aparece evidencia observable de respuesta asociada a la acción; 0 en caso contrario",
        "binaria",
        "ACCION",
        "system_feedback_events"
    ),

    spec(
        "feedback_coverage_ratio",
        "acciones que requieren feedback + feedback detectado",
        "Listeners + MutationObserver",
        "acciones_con_feedback / acciones_evaluables",
        "ratio",
        "EJECUCION",
        "system_feedback_events"
    ),

    spec(
        "feedback_latency_ms",
        "timestamp de acción + timestamp de primer feedback",
        "Listeners + MutationObserver",
        "timestamp_feedback - timestamp_accion",
        "ms",
        "ACCION",
        "system_feedback_events"
    ),

    spec(
        "loader_present",
        "elementos loader/spinner/progress",
        "DOM + MutationObserver + roles ARIA",
        "1 si se detecta loader durante una operación",
        "binaria",
        "ESTADO",
        "system_feedback_events"
    ),

    spec(
        "loader_duration_ms",
        "aparición y desaparición del loader",
        "MutationObserver",
        "timestamp_fin - timestamp_inicio",
        "ms",
        "ESTADO",
        "system_feedback_events"
    ),

    spec(
        "progress_indicator_present",
        "progress, role=progressbar y componentes equivalentes",
        "DOM / ARIA",
        "1 si existe indicador de progreso",
        "binaria",
        "PAGINA",
        "dom_snapshots"
    ),

    spec(
        "status_message_present",
        "role=status, aria-live, role=alert",
        "DOM / ARIA / MutationObserver",
        "1 si existe región de estado observable",
        "binaria",
        "PAGINA",
        "dom_snapshots"
    ),

    spec(
        "completion_feedback_present",
        "mensaje/estado posterior a finalización exitosa",
        "MutationObserver + criterio de finalización",
        "1 si la acción terminada genera feedback observable",
        "binaria",
        "ACCION",
        "system_feedback_events"
    )
]

In [27]:
specifications += [

    spec(
        "component_style_consistency_ratio",
        "tag, role, clases y estilos computados de componentes equivalentes",
        "DOM + getComputedStyle()",
        "componentes equivalentes visualmente consistentes / componentes comparados",
        "ratio",
        "EJECUCION",
        "dom_snapshots"
    ),

    spec(
        "css_class_consistency_ratio",
        "clases CSS de componentes semánticamente equivalentes",
        "DOM",
        "grupos consistentes / grupos comparados",
        "ratio",
        "EJECUCION",
        "dom_snapshots"
    ),

    spec(
        "navigation_consistency_ratio",
        "estructura y orden de elementos de navegación entre páginas",
        "DOM snapshots",
        "estados con navegación consistente / estados comparados",
        "ratio",
        "EJECUCION",
        "dom_snapshots"
    ),

    spec(
        "label_consistency_ratio",
        "texto/nombre accesible de controles equivalentes",
        "DOM / ARIA",
        "controles equivalentes con etiquetado consistente / controles comparados",
        "ratio",
        "EJECUCION",
        "dom_snapshots"
    ),

    spec(
        "semantic_role_consistency_ratio",
        "roles HTML/ARIA de elementos funcionalmente equivalentes",
        "DOM / ARIA",
        "roles consistentes / elementos equivalentes comparados",
        "ratio",
        "EJECUCION",
        "dom_snapshots"
    ),

    spec(
        "dom_structure_similarity",
        "representación estructural normalizada del DOM",
        "DOM snapshots",
        "similitud estructural entre estados comparables",
        "ratio",
        "EJECUCION",
        "dom_snapshots",
        "La métrica concreta de similitud deberá quedar fijada antes del experimento."
    )
]

In [28]:
specifications += [

    spec(
        "input_validation_present",
        "required, pattern, min, max, minlength, maxlength, type",
        "DOM",
        "1 si existen mecanismos declarativos de validación",
        "binaria",
        "PAGINA",
        "dom_snapshots"
    ),

    spec(
        "client_validation_present",
        "eventos invalid + cambios de validación visibles",
        "DOM events + MutationObserver",
        "1 si se detecta validación antes del envío al servidor",
        "binaria",
        "FORMULARIO",
        "validation_events"
    ),

    spec(
        "validation_error_count",
        "eventos invalid/errores de validación",
        "document.addEventListener('invalid', ..., true)",
        "conteo de errores de validación observados",
        "eventos",
        "EJECUCION",
        "validation_events"
    ),

    spec(
        "invalid_submission_blocked",
        "submit intent + resultado de validación",
        "submit/invalid listeners",
        "1 cuando intento inválido es impedido antes de completar envío",
        "binaria",
        "ACCION",
        "validation_events"
    ),

    spec(
        "confirmation_dialog_present",
        "diálogo previo a acciones relevantes",
        "DOM + dialog/role=dialog + eventos",
        "1 si existe confirmación antes de acción definida como crítica",
        "binaria",
        "ACCION",
        "error_prevention_events"
    ),

    spec(
        "destructive_action_protected",
        "acción destructiva + mecanismo de confirmación/reversión",
        "DOM + eventos",
        "1 si acciones destructivas definidas poseen protección",
        "binaria",
        "ACCION",
        "error_prevention_events"
    ),

    spec(
        "error_feedback_present",
        "mensaje posterior a error",
        "MutationObserver + regiones de error",
        "1 si el error genera feedback observable",
        "binaria",
        "ERROR",
        "validation_events"
    ),

    spec(
        "user_error_count",
        "acciones clasificadas según criterios predefinidos de error",
        "eventos + reglas de tarea",
        "conteo de errores operacionales",
        "eventos",
        "EJECUCION",
        "validation_events",
        "Necesita una definición operacional previa; no contar automáticamente todos los clics repetidos como errores."
    )
]

In [29]:
specifications += [

    spec(
        "accessible_name_ratio",
        "nombre accesible de controles interactivos",
        "DOM / ARIA",
        "controles con nombre accesible / controles interactivos",
        "ratio",
        "PAGINA",
        "dom_snapshots"
    ),

    spec(
        "visible_labels_ratio",
        "etiquetas realmente visibles de controles",
        "DOM + estilos computados",
        "controles con etiqueta visible / controles evaluables",
        "ratio",
        "PAGINA",
        "dom_snapshots"
    ),

    spec(
        "unlabeled_controls_count",
        "controles sin etiqueta/nombre apropiado",
        "DOM / ARIA",
        "conteo",
        "controles",
        "PAGINA",
        "dom_snapshots"
    ),

    spec(
        "breadcrumbs_present",
        "breadcrumb semántico/visual",
        "DOM / ARIA",
        "1 si existe breadcrumb identificable",
        "binaria",
        "PAGINA",
        "dom_snapshots"
    ),

    spec(
        "navigation_persistence_ratio",
        "presencia de navegación crítica entre estados",
        "DOM snapshots",
        "estados con navegación crítica visible / estados visitados",
        "ratio",
        "EJECUCION",
        "dom_snapshots"
    ),

    spec(
        "navigation_depth",
        "jerarquía de navegación atravesada",
        "navigation logger",
        "máxima profundidad de ruta funcional durante la tarea",
        "niveles",
        "EJECUCION",
        "navigation_events",
        "No confundir con profundidad del árbol DOM."
    ),

    spec(
        "visible_options_ratio",
        "opciones disponibles y perceptibles",
        "DOM + estilos computados",
        "opciones visibles / opciones evaluables",
        "ratio",
        "PAGINA",
        "dom_snapshots"
    )
]

In [30]:
specifications += [

    spec(
        "task_completed",
        "criterio predefinido de éxito de tarea",
        "Task controller / reglas del experimento",
        "1 cuando se alcanza el estado objetivo definido antes de la ejecución",
        "binaria",
        "EJECUCION",
        "sessions",
        "El criterio de éxito debe definirse para cada tarea antes de recolectar datos."
    )
]

In [31]:
for heuristic in [
    "H1_violation",
    "H4_violation",
    "H5_violation",
    "H6_violation"
]:

    specifications.append(
        spec(
            heuristic,
            "evaluación independiente del experto",
            "Formulario de evaluación heurística",
            "Consolidación de evaluadores",
            "binaria",
            "EJECUCION",
            "expert_annotations",
            "1 = violación presente; 0 = violación no identificada."
        )
    )

In [32]:
collection_spec = pd.DataFrame(
    specifications
)

print(
    collection_spec.shape
)

display(
    collection_spec
)

(48, 8)


,variable,raw_signal,capture_mechanism,calculation,unit,granularity,raw_storage,notes
0,instance_id,Identificador generado al iniciar una ejecución,crypto.randomUUID(),Se genera una vez por ejecución usuario-tarea,UUID,EJECUCION,sessions,No se usa como predictor.
1,platform,hostname / origen de la página,window.location.hostname,Normalización del dominio,categorica,EJECUCION,sessions,
2,task_type,Identificador/tipo de tarea experimental,Configuración de la sesión de evaluación,Asignado al iniciar la tarea,categorica,EJECUCION,sessions,
3,page_type,URL + estructura de página,window.location + clasificación,Clasificación de página según reglas de la plataforma,categorica,PAGINA,page_states,No debe inferirse únicamente del pathname sin validación.
4,task_time_s,"task_start_timestamp, task_end_timestamp",performance.now() / Date.now(),(task_end - task_start) / 1000,s,EJECUCION,sessions,
5,mean_response_time_ms,"requestStart, responseEnd",PerformanceObserver / Resource Timing API,media(responseEnd - requestStart),ms,EJECUCION,performance_events,Tiempo de respuesta técnico de recursos/red; no confundir con feedback de interfaz.
6,max_response_time_ms,"requestStart, responseEnd",PerformanceObserver / Resource Timing API,max(responseEnd - requestStart),ms,EJECUCION,performance_events,
7,mean_pre_action_pause_s,timestamps consecutivos de acciones,Listeners de interacción,media(timestamp_accion - timestamp_evento_previo) / 1000,s,EJECUCION,interaction_events,Es una pausa conductual; no debe interpretarse directamente como confusión.
8,click_count,eventos click,"document.addEventListener('click', ...)",conteo de eventos click válidos,eventos,EJECUCION,interaction_events,
9,keypress_count,eventos keydown,"document.addEventListener('keydown', ...)",conteo de pulsaciones,eventos,EJECUCION,interaction_events,No almacenar el contenido escrito por razones de privacidad; solo metadatos necesarios.


In [33]:
missing = sorted(
    set(dictionary["variable"])
    -
    set(collection_spec["variable"])
)

missing

[]

In [34]:
collection_spec[
    "variable"
].duplicated().sum()

np.int64(0)

In [35]:
collection_spec = (
    dictionary
    .merge(
        collection_spec,
        on="variable",
        how="left",
        validate="one_to_one"
    )
)

In [36]:
collection_spec.shape

(48, 17)

In [37]:
def fuente_final(row):

    if row["rol_ml"] == "Y":
        return "EXPERTO"

    if row["captura_extension"] == "DIRECTA":
        return "EXTENSION_DIRECTA"

    if row["captura_extension"] == "DERIVADA":
        return "EXTENSION_DERIVADA"

    return "REVISAR"

In [38]:
collection_spec[
    "final_source"
] = collection_spec.apply(
    fuente_final,
    axis=1
)

In [39]:
collection_spec[
    "final_source"
].value_counts()

final_source
EXTENSION_DIRECTA     22
EXTENSION_DERIVADA    22
EXPERTO                4
Name: count, dtype: int64

In [40]:
collection_spec[
    collection_spec["categoria"].isin(
        ["H1", "H4", "H5", "H6"]
    )
][
    [
        "variable",
        "categoria",
        "rol_ml",
        "capture_mechanism",
        "calculation",
        "unit",
        "granularity",
        "final_source"
    ]
]

,variable,categoria,rol_ml,capture_mechanism,calculation,unit,granularity,final_source
14,feedback_present,H1,X,MutationObserver,1 si aparece evidencia observable de respuesta asociada a la acción; 0 en caso contrario,binaria,ACCION,EXTENSION_DIRECTA
15,feedback_coverage_ratio,H1,X,Listeners + MutationObserver,acciones_con_feedback / acciones_evaluables,ratio,EJECUCION,EXTENSION_DERIVADA
16,feedback_latency_ms,H1,X,Listeners + MutationObserver,timestamp_feedback - timestamp_accion,ms,ACCION,EXTENSION_DIRECTA
17,loader_present,H1,X,DOM + MutationObserver + roles ARIA,1 si se detecta loader durante una operación,binaria,ESTADO,EXTENSION_DIRECTA
18,loader_duration_ms,H1,X,MutationObserver,timestamp_fin - timestamp_inicio,ms,ESTADO,EXTENSION_DIRECTA
19,progress_indicator_present,H1,X,DOM / ARIA,1 si existe indicador de progreso,binaria,PAGINA,EXTENSION_DIRECTA
20,status_message_present,H1,X,DOM / ARIA / MutationObserver,1 si existe región de estado observable,binaria,PAGINA,EXTENSION_DIRECTA
21,completion_feedback_present,H1,X,MutationObserver + criterio de finalización,1 si la acción terminada genera feedback observable,binaria,ACCION,EXTENSION_DIRECTA
22,component_style_consistency_ratio,H4,X,DOM + getComputedStyle(),componentes equivalentes visualmente consistentes / componentes comparados,ratio,EJECUCION,EXTENSION_DERIVADA
23,css_class_consistency_ratio,H4,X,DOM,grupos consistentes / grupos comparados,ratio,EJECUCION,EXTENSION_DERIVADA


In [41]:
raw_tables = pd.DataFrame([
    {
        "tabla": "sessions",
        "descripcion": "Una fila por ejecución usuario-tarea",
        "clave": "instance_id"
    },

    {
        "tabla": "interaction_events",
        "descripcion": "Clicks, teclado, scroll y acciones del usuario",
        "clave": "event_id"
    },

    {
        "tabla": "navigation_events",
        "descripcion": "Cambios de URL/estado/navegación",
        "clave": "event_id"
    },

    {
        "tabla": "performance_events",
        "descripcion": "Métricas de rendimiento y respuesta de recursos",
        "clave": "event_id"
    },

    {
        "tabla": "system_feedback_events",
        "descripcion": "Feedback, loaders, mensajes y latencias",
        "clave": "event_id"
    },

    {
        "tabla": "validation_events",
        "descripcion": "Intentos inválidos y errores de formularios",
        "clave": "event_id"
    },

    {
        "tabla": "error_prevention_events",
        "descripcion": "Confirmaciones y protección de acciones críticas",
        "clave": "event_id"
    },

    {
        "tabla": "dom_snapshots",
        "descripcion": "Características estructurales del estado de interfaz",
        "clave": "snapshot_id"
    },

    {
        "tabla": "expert_annotations",
        "descripcion": "Etiquetas H1/H4/H5/H6 proporcionadas por expertos",
        "clave": "annotation_id"
    }
])

raw_tables

,tabla,descripcion,clave
0,sessions,Una fila por ejecución usuario-tarea,instance_id
1,interaction_events,"Clicks, teclado, scroll y acciones del usuario",event_id
2,navigation_events,Cambios de URL/estado/navegación,event_id
3,performance_events,Métricas de rendimiento y respuesta de recursos,event_id
4,system_feedback_events,"Feedback, loaders, mensajes y latencias",event_id
5,validation_events,Intentos inválidos y errores de formularios,event_id
6,error_prevention_events,Confirmaciones y protección de acciones críticas,event_id
7,dom_snapshots,Características estructurales del estado de interfaz,snapshot_id
8,expert_annotations,Etiquetas H1/H4/H5/H6 proporcionadas por expertos,annotation_id


In [42]:
collection_spec.to_csv(
    REPORTS_DIR
    / "data_collection_specification.csv",
    index=False
)

raw_tables.to_csv(
    REPORTS_DIR
    / "extension_raw_tables.csv",
    index=False
)

In [43]:
print(
    (
        REPORTS_DIR
        / "data_collection_specification.csv"
    ).exists()
)

print(
    (
        REPORTS_DIR
        / "extension_raw_tables.csv"
    ).exists()
)

True
True
